# YouTube Chatbot RAG
URL -> transcript -> chunks -> embeddings -> FAISS -> retrieval -> prompt -> LLM -> parser

In [14]:
!pip install -q youtube-transcript-api>=1.0.0 langchain-community langchain-core langchain-text-splitters langchain-huggingface langchain-groq faiss-cpu sentence-transformers

The process cannot access the file because it is being used by another process.


In [15]:
import os, getpass
# API key kabhi notebook mein hardcode na karein. Yahan runtime par poochi jati hai.
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")

In [16]:
import re
from operator import itemgetter
from urllib.parse import urlparse, parse_qs

from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

ModuleNotFoundError: No module named 'youtube_transcript_api'

## STEP 0: URL se video id

In [ ]:
_ID_RE = re.compile(r"^[A-Za-z0-9_-]{11}$")

def extract_video_id(url_or_id):
    text = url_or_id.strip()
    if _ID_RE.match(text):
        return text
    if "://" not in text:
        text = "https://" + text
    parsed = urlparse(text)
    host = (parsed.hostname or "").lower()
    video_id = None
    if host == "youtu.be":
        video_id = parsed.path.lstrip("/").split("/")[0]
    elif host.endswith("youtube.com") or host.endswith("youtube-nocookie.com"):
        if parsed.path == "/watch":
            video_id = parse_qs(parsed.query).get("v", [None])[0]
        else:
            parts = [p for p in parsed.path.split("/") if p]
            if len(parts) >= 2 and parts[0] in {"embed", "shorts", "live", "v"}:
                video_id = parts[1]
    if video_id and _ID_RE.match(video_id):
        return video_id
    raise ValueError("Valid YouTube URL nahi hai")

url = "https://www.youtube.com/watch?v=Rni7Fz7208c"   # <- yahan apna URL dalein
video_id = extract_video_id(url)
video_id

## STEP 1: DOCUMENT INGESTION

In [ ]:
# Transcript timestamps ke sath tukron mein aata hai, hum unhe join kar ke aik string bana dete hain
try:
    ytt_api = YouTubeTranscriptApi()
    try:
        fetched_transcript = ytt_api.fetch(video_id, languages=["en", "hi", "ur"])
    except NoTranscriptFound:
        # Pasandida language na mile to jo pehli available ho wo le lo
        fetched_transcript = next(iter(ytt_api.list(video_id))).fetch()
    transcript = " ".join(snippet.text for snippet in fetched_transcript)
    print(transcript[:500], "...")
except TranscriptsDisabled:
    print("No captions available for this video.")

## TEXT SPLITTER

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
print("Total chunks:", len(chunks))
chunks[0]

## EMBEDDING AND VECTOR STORE

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)
print("Total documents:", len(vector_store.index_to_docstore_id))

## RETRIEVAL

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})
retriever.invoke("how to regulate ai")

## AUGMENTATION

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

prompt = PromptTemplate(
    template="""You are a helpful assistant answering questions about a YouTube video.
Answer ONLY from the transcript context below.
If the context is insufficient, say you don't know. Do not make things up.
Reply in the same language the user writes in.

Transcript context:
{context}

Chat history:
{history}

Question: {question}

Answer:""",
    input_variables=["context", "history", "question"],
)

In [ ]:
question = "Is AI risk discussed in this video? If yes, what was said?"
retrieved_docs = retriever.invoke(question)

# Retriever list of Documents deta hai, prompt ke liye unka page_content join karte hain
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

final_prompt = prompt.invoke({"context": context_text, "history": "No previous messages.", "question": question})
final_prompt

## GENERATION

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

## CHAIN
Ab har step baar baar call nahi karna parta: aik `invoke` mein poora pipeline chalta hai.

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

parallel_chain = RunnableParallel({
    "context": itemgetter("question") | retriever | RunnableLambda(format_docs),
    "question": itemgetter("question"),
    "history": itemgetter("history"),
})

parser = StrOutputParser()
final_chain = parallel_chain | prompt | llm | parser

final_chain.invoke({"question": "Can you summarize this video into five points?", "history": "No previous messages."})

## Streamlit app
UI ke liye `app.py` aur `rag_pipeline.py` use karein (`streamlit run app.py`).